# Meridian health data cleanning project

In [ ]:
#import dependices

import pandas as pd
import numpy as np
import re
!pip install rapidfuzz
import rapidfuzz
from rapidfuzz.fuzz import ratio
import networkx as nx
import matplotlib.pyplot as plt


In [ ]:

#read all data files

categories = pd.read_csv("/content/categories.csv")
customers = pd.read_csv("/content/customers_master.csv")
employees = pd.read_csv ("/content/employees.csv")
prescriptions = pd.read_csv("/content/prescriptions_log.csv")
products = pd.read_csv("/content/products_catalog.csv")
stock = pd.read_csv("/content/stock_condition_log.csv")
suppliers = pd.read_csv("/content/suppliers.csv")
stores = pd.read_csv("/content/stores.csv")
sales_g1 = pd.read_csv("/content/sales_group1_storesA.csv")
sales_g2 = pd.read_csv("/content/sales_group2_storesB.csv")
sales_g3 = pd.read_csv("/content/sales_group3_storesC.csv")


# Customers Table

In [ ]:
#describe customers table

description = pd.DataFrame({
    "Column": customers.columns,
    "Data Type": customers.dtypes.astype(str).values,
    "Non-Null": customers.notna().sum().values,
    "Null": customers.isna().sum().values,
    "Null %": (customers.isna().mean() * 100).round(2).values,
    "Unique": customers.nunique().values
})

description

,Column,Data Type,Non-Null,Null,Null %,Unique
0,customer_id,int64,1555,0,0.00,1555
1,full_name,object,1555,0,0.00,602
2,email,object,1555,0,0.00,1555
3,phone,object,1555,0,0.00,1513
4,city,object,1555,0,0.00,45
5,signup_date,object,1555,0,0.00,1132
6,notes,object,696,859,55.24,25


In [ ]:
#data types change
customers = customers.astype({
    "customer_id": "int64",
    "full_name": "string",
    "email": "string",
    "phone": "string",
    "city": "string",
    "notes": "string"
})

customers["signup_date"] = pd.to_datetime(
    customers["signup_date"],
    errors="coerce"
)

customers.dtypes

,0
customer_id,int64
full_name,string[python]
email,string[python]
phone,string[python]
city,string[python]
signup_date,datetime64[ns]
notes,string[python]


In [ ]:
#lowering and striping  string values

customers[customers.select_dtypes(include="string").columns] = (
    customers.select_dtypes(include="string")
    .apply(lambda col: col.str.lower().str.strip())
)


In [ ]:
customers["city"].unique()

<StringArray>
[     'montreal',      'edmonton',       'moncton',        'regina',
      'waterloo',      'winnipeg',       'toronto',     'vancouver',
    'regina, sk',       'calgary',  'winnipeg, mb',   'quebec city',
     'kitchener',     'saskatoon',        'london',   'calgary, ab',
        'ottawa', 'vancouver, bc',       'halifax',      'vancouvr',
    'ottawa, on',  'montreal, qc',   'halifax, ns',      'hamilton',
   'toronto, on',      'montréal',  'waterloo, on',        'quebec',
  'edmonton, ab',       'qc city',         'trnto',   'moncton, nb']
Length: 32, dtype: string

In [ ]:
#mapping city names for remove duplicat meaning

city_mapping = {
    "toronto": "toronto",
    "toronto, on": "toronto",
    "trnto": "toronto",

    "vancouver": "vancouver",
    "vancouver, bc": "vancouver",
    "vancouvr": "vancouver",

    "montreal": "montreal",
    "montréal": "montreal",
    "montreal, qc": "montreal",

    "calgary": "calgary",
    "calgary, ab": "calgary",

    "winnipeg": "winnipeg",
    "winnipeg, mb": "winnipeg",

    "regina": "regina",
    "regina, sk": "regina",

    "ottawa": "ottawa",
    "ottawa, on": "ottawa",

    "halifax": "halifax",
    "halifax, ns": "halifax",

    "edmonton": "edmonton",
    "edmonton, ab": "edmonton",

    "waterloo": "waterloo",
    "waterloo, on": "waterloo",

    "moncton": "moncton",
    "moncton, nb": "moncton",

    "quebec city": "quebec",
    "qc city": "quebec",
    "quebec": "quebec"
}

customers["city"] = customers["city"].replace(city_mapping)

In [ ]:
customers["city"].unique()

<StringArray>
[ 'montreal',  'edmonton',   'moncton',    'regina',  'waterloo',  'winnipeg',
   'toronto', 'vancouver',   'calgary',    'quebec', 'kitchener', 'saskatoon',
    'london',    'ottawa',   'halifax',  'hamilton']
Length: 16, dtype: string

In [ ]:
# loyalty extaction

def extract_loyalty(note):
    if pd.isna(note) or note == "":
        return pd.Series({"loyalty_tier": None, "loyalty_since": None})

    m = re.search(r"(silver|gold|platinum) member - loyalty since (\d{4})", note)
    if m:
        return pd.Series({"loyalty_tier": m.group(1), "loyalty_since": int(m.group(2))})
    return pd.Series({"loyalty_tier": None, "loyalty_since": None})

loyalty = customers["notes"].apply(extract_loyalty)
loyalty = loyalty.astype({
    "loyalty_tier": "string",
    "loyalty_since": "Int64"
})

customers = pd.concat([customers, loyalty], axis=1)

In [ ]:
#phone cleaning
customers["phone"] = customers["phone"].astype(str).str.replace(r"\D", "", regex=True)

In [ ]:
df= customers.copy()
blocks = df.groupby("city").groups

G = nx.Graph()
G.add_nodes_from(df.index)

threshold = 80

for city, indices in blocks.items():
    indices = list(indices)

    for i in range(len(indices)):
        for j in range(i + 1, len(indices)):

            idx_a = indices[i]
            idx_b = indices[j]

            a = df.loc[idx_a]
            b = df.loc[idx_b]

            name_score = ratio(a["full_name"], b["full_name"])
            email_score = ratio(a["email"], b["email"])

            phone_score = (
                100
                if a["phone"] and b["phone"] and a["phone"] == b["phone"]
                else 0
            )

            score = (
                name_score * 0.40 +
                email_score * 0.40 +
                phone_score * 0.20
            )

            if score >= threshold:
                G.add_edge(
                    idx_a,
                    idx_b,
                    score=round(score, 2)
                )


groups = [
    list(group)
    for group in nx.connected_components(G)
    if len(group) > 1
]


results = []

for group_id, group in enumerate(groups, start=1):

    group_scores = []

    for i in range(len(group)):
        for j in range(i + 1, len(group)):

            if G.has_edge(group[i], group[j]):
                score = G[group[i]][group[j]]["score"]
                group_scores.append(score)

    group_score = round(
        sum(group_scores) / len(group_scores),
        2
    )

    for idx in group:
        results.append({
            "group_id": group_id,
            "customer_id": df.loc[idx, "customer_id"],
            "group_size": len(group),
            "group_score": group_score
        })


groups_df = pd.DataFrame(results)




In [ ]:
groups_df

,group_id,customer_id,group_size,group_score
0,1,1501,2,84.55
1,1,86,2,84.55
2,2,518,2,85.33
3,2,1506,2,85.33
4,3,490,2,92.20
...,...,...,...,...
79,40,837,2,85.30
80,41,1363,2,90.62
81,41,1536,2,90.62
82,42,220,2,86.33


In [ ]:
matched = (
    groups_df
    .merge(
        df[["customer_id", "full_name", "email", "phone", "city"]],
        on="customer_id",
        how="left"
    )
)

result = matched.merge(
    matched,
    on="group_id",
    suffixes=("_1", "_2")
)

result = result[
    result["customer_id_1"] < result["customer_id_2"]
]

result = result[
    [
        "group_id",
        "customer_id_1",
        "customer_id_2",
        "full_name_1",
        "full_name_2",
        "email_1",
        "email_2",
        "phone_1",
        "phone_2",
        "city_1",
        "city_2",
        "group_score_1"
    ]
].rename(
    columns={"group_score_1": "group_score"}
)

result

,group_id,customer_id_1,customer_id_2,full_name_1,full_name_2,email_1,email_2,phone_1,phone_2,city_1,city_2,group_score
2,1,86,1501,nancy taylor,nancy taaylor,nancy_taylor86@gmail.com,nancytaaylor1501@outlook.com,7821678948,7821678948,montreal,montreal,84.55
5,2,518,1506,kevin hall,kevin halll,kevin_hall518@gmail.com,kevinhalll1506@yahoo.com,7630544575,7630544575,moncton,moncton,85.33
9,3,490,1511,ryan anderson,ryyan anderson,ryan_anderson490@outlook.com,ryyananderson1511@outlook.com,1387506072,1387506072,waterloo,waterloo,92.20
14,4,159,1514,patricia chan,patricia chan,patricia_chan159@yahoo.com,patricia.chan1514@gmail.com,9507472950,9507472950,calgary,calgary,90.19
17,5,1288,1551,justin tremblay,justinn tremblay,justin_tremblay1288@hotmail.com,justinntremblay1551@gmail.com,1070983552,1070983552,waterloo,waterloo,90.71
22,6,1262,1523,matthew macdonald,matthew macdonnald,matthewmacdonald1262@gmail.com,matthew.macdonnald1523@hotmail.com,0346718657,0346718657,vancouver,vancouver,92.61
26,7,92,1542,nancy hall,nancy hall,nancy_hall92@yahoo.com,nancy.hall1542@hotmail.com,2082302039,2082302039,quebec,quebec,88.33
30,8,411,1534,kevin young,keevin young,kevin.young411@outlook.com,keevin_young1534@yahoo.com,6819081876,6819081876,regina,regina,85.95
34,9,43,1532,susan clark,susan cllark,susanclark43@yahoo.com,susan_cllark1532@gmail.com,2519836473,2519836473,toronto,toronto,86.59
38,10,872,1543,laura lee,laura leee,laura.lee872@hotmail.com,laura.leee1543@outlook.com,2861539636,2861539636,ottawa,ottawa,85.09


# Sales Table

In [ ]:

#standeraized the table columns names

sales_g1 = sales_g1.rename(columns={
    "quantity": "quantity",
    "unit_price": "unit_price"
})

sales_g2 = sales_g2.rename(columns={
    "qty": "quantity",
    "amount": "unit_price"
})

sales_g3 = sales_g3.rename(columns={
    "quantity": "quantity",
    "unit_price": "unit_price"
})

# add columns notes into g1 & g2

sales_g1["notes"] = pd.NA
sales_g2["notes"] = pd.NA



In [ ]:
# order columns for each group

columns = [
    "sale_id",
    "store_id",
    "employee_id",
    "product_id",
    "customer_id",
    "quantity",
    "sale_date",
    "unit_price",
    "discount_pct",
    "payment_method",
    "notes"
]

sales_g1 = sales_g1[columns]
sales_g2 = sales_g2[columns]
sales_g3 = sales_g3[columns]

In [ ]:
#combining 3 tables into 1 sales table to start clean

sales = pd.concat(
    [sales_g1, sales_g2, sales_g3],
    ignore_index=True
)

In [ ]:
# validate combination
source_total = len(sales_g1) + len(sales_g2) + len(sales_g3)
combined_total = len(sales)

print("Source Total:", source_total)
print("Combined Total:", combined_total)
print("Match:", source_total == combined_total)

Source Total: 9950
Combined Total: 9950
Match: True


In [ ]:
# describe sales table

description = pd.DataFrame({
    "Column": sales.columns,
    "Data Type": sales.dtypes.astype(str).values,
    "Non-Null": sales.notna().sum().values,
    "Null": sales.isna().sum().values,
    "Null %": (sales.isna().mean() * 100).round(2).values,
    "Unique": sales.nunique().values
})

description

,Column,Data Type,Non-Null,Null,Null %,Unique
0,sale_id,int64,9950,0,0.00,9900
1,store_id,int64,9950,0,0.00,12
2,employee_id,int64,9950,0,0.00,120
3,product_id,int64,9950,0,0.00,4300
4,customer_id,int64,9950,0,0.00,1456
5,quantity,int64,9950,0,0.00,5
6,sale_date,object,9950,0,0.00,870
7,unit_price,object,9950,0,0.00,5768
8,discount_pct,float64,9950,0,0.00,212
9,payment_method,object,9950,0,0.00,12


In [ ]:
#data types change

sales["unit_price"] = (
    sales["unit_price"]
    .astype("string")
    .str.replace("$", "", regex=False)
    .str.strip()
)

sales["unit_price"] = pd.to_numeric(
    sales["unit_price"],
    errors="coerce"
)

sales = sales.astype({
    "sale_id": "int64",
    "store_id": "int64",
    "employee_id": "int64",
    "product_id": "int64",
    "customer_id": "int64",
    "quantity": "int64",
    "unit_price": "float64",
    "discount_pct": "float64",
    "payment_method": "string",
    "notes": "string"
})

sales["sale_date"] = (
    sales["sale_date"]
    .astype("string")
    .str.strip()
)

sales["sale_date"] = pd.to_datetime(
    sales["sale_date"],
    format="mixed",
    errors="coerce"
)

In [ ]:
# describe sales table

description = pd.DataFrame({
    "Column": sales.columns,
    "Data Type": sales.dtypes.astype(str).values,
    "Non-Null": sales.notna().sum().values,
    "Null": sales.isna().sum().values,
    "Null %": (sales.isna().mean() * 100).round(2).values,
    "Unique": sales.nunique().values
})

description

,Column,Data Type,Non-Null,Null,Null %,Unique
0,sale_id,int64,9950,0,0.00,9900
1,store_id,int64,9950,0,0.00,12
2,employee_id,int64,9950,0,0.00,120
3,product_id,int64,9950,0,0.00,4300
4,customer_id,int64,9950,0,0.00,1456
5,quantity,int64,9950,0,0.00,5
6,sale_date,datetime64[ns],9950,0,0.00,464
7,unit_price,float64,9950,0,0.00,3958
8,discount_pct,float64,9950,0,0.00,212
9,payment_method,string,9950,0,0.00,12


In [ ]:
#lowering and striping  string values

sales[sales.select_dtypes(include="string").columns] = (
    sales.select_dtypes(include="string")
    .apply(lambda col: col.str.lower().str.strip())
)


In [ ]:
def extract_promo(row):
    note = row["notes"]
    discount = row["discount_pct"]

    promo_code = None

    if pd.notna(note) and note.strip() != "":
        m = re.search(r"promo code (\w+) applied at checkout", note)

        if m:
            promo_code = m.group(1)

    # No discount
    if pd.isna(discount) or discount == 0:
        discount_reason = "no_discount"

    # Discount exists → check the reason
    elif pd.notna(note) and note.strip() != "":
        if re.search(r"promo code (\w+) applied at checkout", note):
            discount_reason = "promo_code"

        elif "competitor" in note.lower():
            discount_reason = "price_match"

        elif "loyalty" in note.lower():
            discount_reason = "loyalty_tier"

        else:
            discount_reason = "other"

    # Discount exists but no note
    else:
        discount_reason = "other"

    return pd.Series({
        "promo_code": promo_code,
        "discount_reason": discount_reason
    })


extracted = sales.apply(extract_promo, axis=1)

sales = pd.concat([sales, extracted], axis=1)

sales

,sale_id,store_id,employee_id,product_id,customer_id,quantity,sale_date,unit_price,discount_pct,payment_method,notes,promo_code,discount_reason
0,1,3,21,3878,99,2,2025-01-20,184.16,8.1,cash,<NA>,None,other
1,7,3,88,465,997,1,2024-10-20,108.23,6.9,cash,<NA>,None,other
2,8,2,32,3997,131,1,2024-07-12,298.25,6.4,debit card,<NA>,None,other
3,12,2,13,4985,151,2,2025-07-26,140.55,5.4,cash,<NA>,None,other
4,13,1,3,3210,205,4,2025-09-03,179.13,32.4,credit card,<NA>,None,other
...,...,...,...,...,...,...,...,...,...,...,...,...,...
9945,7937,10,87,4504,372,3,2025-04-16,109.51,33.7,cash,<NA>,None,other
9946,6711,11,63,2028,805,1,2025-10-16,51.18,26.7,credit card,loyalty tier discount applied,None,loyalty_tier
9947,8450,9,27,26,288,1,2025-04-10,48.36,5.2,credit card,loyalty tier discount applied,None,loyalty_tier
9948,637,10,39,2571,282,1,2025-11-23,115.75,28.0,credit card,promo code welcome15 applied at checkout,welcome15,promo_code


In [ ]:
dataframes = {
    "categories": categories,
    "customers": customers,
    "employees": employees,
    "prescriptions": prescriptions,
    "products": products,
    "stock": stock,
    "suppliers": suppliers,
    "stores": stores,
    "sales": sales
}

columns_dict = {name: df.columns.tolist() for name, df in dataframes.items()}

display(columns_dict)

{'categories': ['category_id', 'category_name'],
 'customers': ['customer_id',
  'full_name',
  'email',
  'phone',
  'city',
  'signup_date',
  'notes',
  'loyalty_tier',
  'loyalty_since'],
 'employees': ['employee_id',
  'store_id',
  'employee_name',
  'email',
  'position',
  'salary',
  'hire_date',
  'is_pharmacist'],
 'prescriptions': ['prescription_id',
  'store_id',
  'product_id',
  'customer_id',
  'doctor_name',
  'issue_date',
  'dosage'],
 'products': ['product_id',
  'product_name',
  'category_id',
  'supplier_id',
  'unit_price',
  'cost_price',
  'brand',
  'expiry_date',
  'storage_condition',
  'created_at',
  'pack_info'],
 'stock': ['stock_id',
  'store_id',
  'product_id',
  'quantity',
  'reorder_level',
  'last_restocked',
  'batch_number',
  'expiry_date',
  'received_date',
  'condition_notes'],
 'suppliers': ['supplier_id',
  'supplier_name',
  'contact_person',
  'phone_number',
  'city'],
 'stores': ['store_id', 'store_name', 'region', 'city', 'manager_na

# categories table

In [ ]:
#data types change
categories = categories.astype({
    "category_id": "int64",
    "category_name": "string"
})

In [ ]:
#lowering and striping  string values

categories[categories.select_dtypes(include="string").columns] = (
    categories.select_dtypes(include="string")
    .apply(lambda col: col.str.lower().str.strip())
)


In [ ]:
# describe table

description = pd.DataFrame({
    "Column": categories.columns,
    "Data Type": categories.dtypes.astype(str).values,
    "Non-Null": categories.notna().sum().values,
    "Null": categories.isna().sum().values,
    "Null %": (categories.isna().mean() * 100).round(2).values,
    "Unique": categories.nunique().values
})

description

,Column,Data Type,Non-Null,Null,Null %,Unique
0,category_id,int64,9,0,0.0,9
1,category_name,string,9,0,0.0,9


# employees table

In [ ]:
#data types change
employees = employees.astype({
    "employee_id": "int64",
    "store_id": "int64",
    "employee_name": "string",
    "email": "string",
    "position": "string",
    "salary": "float64",
    "is_pharmacist": "boolean"
})

employees["hire_date"] = (
    employees["hire_date"]
    .astype("string")
    .str.strip()
)

employees["hire_date"] = pd.to_datetime(
    employees["hire_date"],
    format="mixed",
    errors="coerce"
)

In [ ]:
#lowering and striping  string values

employees[employees.select_dtypes(include="string").columns] = (
    employees.select_dtypes(include="string")
    .apply(lambda col: col.str.lower().str.strip())
)


In [ ]:
# describe table

description = pd.DataFrame({
    "Column": employees.columns,
    "Data Type": employees.dtypes.astype(str).values,
    "Non-Null": employees.notna().sum().values,
    "Null": employees.isna().sum().values,
    "Null %": (employees.isna().mean() * 100).round(2).values,
    "Unique": employees.nunique().values
})

description

,Column,Data Type,Non-Null,Null,Null %,Unique
0,employee_id,int64,120,0,0.0,120
1,store_id,int64,120,0,0.0,12
2,employee_name,string,120,0,0.0,120
3,email,string,120,0,0.0,120
4,position,string,120,0,0.0,5
5,salary,float64,120,0,0.0,120
6,hire_date,datetime64[ns],120,0,0.0,118
7,is_pharmacist,boolean,120,0,0.0,2


In [ ]:
# pharmacist handiling
employees["is_pharmacist"] = employees["position"].str.lower().apply(
    lambda x: 1 if x == "pharmacist" else 0
)

employees.loc[employees["is_pharmacist"] == 1]

,employee_id,store_id,employee_name,email,position,salary,hire_date,is_pharmacist
9,10,2,staff_member_10,staff_member_10@company.com,pharmacist,6057.50,2018-11-11,1
17,18,12,staff_member_18,staff_member_18@company.com,pharmacist,9428.81,2022-02-14,1
19,20,5,staff_member_20,staff_member_20@company.com,pharmacist,4049.85,2020-03-05,1
20,21,3,staff_member_21,staff_member_21@company.com,pharmacist,7029.20,2025-10-08,1
21,22,5,staff_member_22,staff_member_22@company.com,pharmacist,5459.16,2020-02-12,1
45,46,3,staff_member_46,staff_member_46@company.com,pharmacist,6660.27,2020-02-05,1
51,52,9,staff_member_52,staff_member_52@company.com,pharmacist,7526.84,2025-01-04,1
57,58,1,staff_member_58,staff_member_58@company.com,pharmacist,6294.37,2024-08-26,1
60,61,9,staff_member_61,staff_member_61@company.com,pharmacist,8773.88,2018-08-03,1
68,69,2,staff_member_69,staff_member_69@company.com,pharmacist,9944.97,2025-02-26,1


# prescriptions table

In [ ]:
#data types change
prescriptions = prescriptions.astype({
    "prescription_id": "int64",
    "store_id": "int64",
    "product_id": "int64",
    "customer_id": "int64",
    "doctor_name": "string",
    "dosage": "string"
})

prescriptions["issue_date"] = (
    prescriptions["issue_date"]
    .astype("string")
    .str.strip()
)

prescriptions["issue_date"] = pd.to_datetime(
    prescriptions["issue_date"],
    format="mixed",
    errors="coerce"
)

In [ ]:
#lowering and striping  string values

prescriptions[prescriptions.select_dtypes(include="string").columns] = (
    prescriptions.select_dtypes(include="string")
    .apply(lambda col: col.str.lower().str.strip())
)

In [ ]:
# describe table

description = pd.DataFrame({
    "Column": prescriptions.columns,
    "Data Type": prescriptions.dtypes.astype(str).values,
    "Non-Null": prescriptions.notna().sum().values,
    "Null": prescriptions.isna().sum().values,
    "Null %": (prescriptions.isna().mean() * 100).round(2).values,
    "Unique": prescriptions.nunique().values
})

description

,Column,Data Type,Non-Null,Null,Null %,Unique
0,prescription_id,int64,10000,0,0.0,10000
1,store_id,int64,10000,0,0.0,12
2,product_id,int64,10000,0,0.0,566
3,customer_id,int64,10000,0,0.0,1453
4,doctor_name,string,10000,0,0.0,20
5,issue_date,datetime64[ns],10000,0,0.0,435
6,dosage,string,10000,0,0.0,14


#products table



In [ ]:
#data types change
products["unit_price"] = (
    products["unit_price"]
    .astype("string")
    .str.replace("$", "", regex=False)
    .str.strip()
)

products["cost_price"] = (
    products["cost_price"]
    .astype("string")
    .str.replace("$", "", regex=False)
    .str.strip()
)

products["unit_price"] = pd.to_numeric(
    products["unit_price"],
    errors="coerce"
)

products["cost_price"] = pd.to_numeric(
    products["cost_price"],
    errors="coerce"
)

products = products.astype({
    "product_id": "int64",
    "product_name": "string",
    "category_id": "int64",
    "supplier_id": "int64",
    "unit_price": "float64",
    "cost_price": "float64",
    "brand": "string",
    "storage_condition": "string",
    "pack_info": "string"
})

products["expiry_date"] = (
    products["expiry_date"]
    .astype("string")
    .str.strip()
)

products["expiry_date"] = pd.to_datetime(
    products["expiry_date"],
    format="mixed",
    errors="coerce"
)

products["created_at"] = (
    products["created_at"]
    .astype("string")
    .str.strip()
)

products["created_at"] = pd.to_datetime(
    products["created_at"],
    format="mixed",
    errors="coerce"
)

In [ ]:
#lowering and striping  string values

products[products.select_dtypes(include="string").columns] = (
    products.select_dtypes(include="string")
    .apply(lambda col: col.str.lower().str.strip())
)

In [ ]:
# describe table

description = pd.DataFrame({
    "Column": products.columns,
    "Data Type": products.dtypes.astype(str).values,
    "Non-Null": products.notna().sum().values,
    "Null": products.isna().sum().values,
    "Null %": (products.isna().mean() * 100).round(2).values,
    "Unique": products.nunique().values
})

description

,Column,Data Type,Non-Null,Null,Null %,Unique
0,product_id,int64,5000,0,0.0,5000
1,product_name,string,5000,0,0.0,5000
2,category_id,int64,5000,0,0.0,9
3,supplier_id,int64,5000,0,0.0,5
4,unit_price,float64,5000,0,0.0,4548
5,cost_price,float64,5000,0,0.0,4329
6,brand,string,5000,0,0.0,7
7,expiry_date,datetime64[ns],5000,0,0.0,1433
8,storage_condition,string,5000,0,0.0,3
9,created_at,datetime64[ns],5000,0,0.0,1456


# stock table

In [ ]:
#data types change
stock = stock.astype({
    "stock_id": "int64",
    "store_id": "int64",
    "product_id": "int64",
    "quantity": "int64",
    "reorder_level": "int64",
    "batch_number": "string",
    "condition_notes": "string"
})

stock["last_restocked"] = (
    stock["last_restocked"]
    .astype("string")
    .str.strip()
)

stock["last_restocked"] = pd.to_datetime(
    stock["last_restocked"],
    format="mixed",
    errors="coerce"
)

stock["expiry_date"] = (
    stock["expiry_date"]
    .astype("string")
    .str.strip()
)

stock["expiry_date"] = pd.to_datetime(
    stock["expiry_date"],
    format="mixed",
    errors="coerce"
)

stock["received_date"] = (
    stock["received_date"]
    .astype("string")
    .str.strip()
)

stock["received_date"] = pd.to_datetime(
    stock["received_date"],
    format="mixed",
    errors="coerce"
)

In [ ]:
#lowering and striping  string values

stock[stock.select_dtypes(include="string").columns] = (
    stock.select_dtypes(include="string")
    .apply(lambda col: col.str.lower().str.strip())
)

In [ ]:
# describe table

description = pd.DataFrame({
    "Column": stock.columns,
    "Data Type": stock.dtypes.astype(str).values,
    "Non-Null": stock.notna().sum().values,
    "Null": stock.isna().sum().values,
    "Null %": (stock.isna().mean() * 100).round(2).values,
    "Unique": stock.nunique().values
})

description

,Column,Data Type,Non-Null,Null,Null %,Unique
0,stock_id,int64,10000,0,0.00,10000
1,store_id,int64,10000,0,0.00,12
2,product_id,int64,10000,0,0.00,4318
3,quantity,int64,10000,0,0.00,120
4,reorder_level,int64,10000,0,0.00,45
5,last_restocked,datetime64[ns],10000,0,0.00,460
6,batch_number,string,10000,0,0.00,10000
7,expiry_date,datetime64[ns],10000,0,0.00,1402
8,received_date,datetime64[ns],10000,0,0.00,460
9,condition_notes,string,4094,5906,59.06,935


# suppliers table

In [ ]:
#data types change
suppliers = suppliers.astype({
    "supplier_id": "int64",
    "supplier_name": "string",
    "contact_person": "string",
    "phone_number": "string",
    "city": "string"
})

In [ ]:
#lowering and striping  string values

suppliers[suppliers.select_dtypes(include="string").columns] = (
    suppliers.select_dtypes(include="string")
    .apply(lambda col: col.str.lower().str.strip())
)

In [ ]:
# describe table

description = pd.DataFrame({
    "Column": suppliers.columns,
    "Data Type": suppliers.dtypes.astype(str).values,
    "Non-Null": suppliers.notna().sum().values,
    "Null": suppliers.isna().sum().values,
    "Null %": (suppliers.isna().mean() * 100).round(2).values,
    "Unique": suppliers.nunique().values
})

description

,Column,Data Type,Non-Null,Null,Null %,Unique
0,supplier_id,int64,5,0,0.0,5
1,supplier_name,string,5,0,0.0,5
2,contact_person,string,5,0,0.0,5
3,phone_number,string,5,0,0.0,5
4,city,string,5,0,0.0,5


# stores table

In [ ]:
#data types change
stores = stores.astype({
    "store_id": "int64",
    "store_name": "string",
    "region": "string",
    "city": "string",
    "manager_name": "string"
})

In [ ]:
#lowering and striping  string values

stores[stores.select_dtypes(include="string").columns] = (
    stores.select_dtypes(include="string")
    .apply(lambda col: col.str.lower().str.strip())
)

In [ ]:
# describe table

description = pd.DataFrame({
    "Column": stores.columns,
    "Data Type": stores.dtypes.astype(str).values,
    "Non-Null": stores.notna().sum().values,
    "Null": stores.isna().sum().values,
    "Null %": (stores.isna().mean() * 100).round(2).values,
    "Unique": stores.nunique().values
})

description

,Column,Data Type,Non-Null,Null,Null %,Unique
0,store_id,int64,12,0,0.0,12
1,store_name,string,12,0,0.0,12
2,region,string,12,0,0.0,5
3,city,string,12,0,0.0,12
4,manager_name,string,12,0,0.0,12


# End of cleaning

# export cleanned data

In [ ]:
# import os
# import zipfile

# tables = {
#     "categories": categories,
#     "customers": customers,
#     "employees": employees,
#     "prescriptions": prescriptions,
#     "products": products,
#     "stock": stock,
#     "suppliers": suppliers,
#     "stores": stores,
#     "sales": sales
# }

# folder = "Meridian Health Cleaned Data"
# os.makedirs(folder, exist_ok=True)

# for name, df in tables.items():
#     df.to_csv(
#         f"{folder}/{name}.csv",
#         index=False
#     )

# with zipfile.ZipFile(
#     "Meridian Health Cleaned Data.zip",
#     "w",
#     zipfile.ZIP_DEFLATED
# ) as zip_file:

#     for file in os.listdir(folder):
#         zip_file.write(
#             f"{folder}/{file}",
#             arcname=file
#         )

In [ ]:
sales = sales.merge(
    products[['product_id', 'cost_price']],
    on='product_id',
    how='left'
)

sales['discount_pct'] = sales['discount_pct'].fillna(0)

sales['total_amount'] = (
    sales['quantity'] * sales['unit_price']
    * (1 - sales['discount_pct'] / 100)
).round(2)

sales['total_cost'] = (
    sales['quantity'] * sales['cost_price']
).round(2)

sales['profit'] = (
    sales['total_amount'] - sales['total_cost']
).round(2)

sales['margin'] = (
    sales['profit'] / sales['total_amount'] * 100
).where(
    sales['total_amount'] != 0
).round(2)

In [ ]:
sales

,sale_id,store_id,employee_id,product_id,customer_id,quantity,sale_date,unit_price,discount_pct,payment_method,notes,promo_code,discount_reason,cost_price,total_amount,total_cost,profit,margin
0,1,3,21,3878,99,2,2025-01-20,184.16,8.1,cash,<NA>,None,other,81.69,338.49,163.38,175.11,51.73
1,7,3,88,465,997,1,2024-10-20,108.23,6.9,cash,<NA>,None,other,57.85,100.76,57.85,42.91,42.59
2,8,2,32,3997,131,1,2024-07-12,298.25,6.4,debit card,<NA>,None,other,212.63,279.16,212.63,66.53,23.83
3,12,2,13,4985,151,2,2025-07-26,140.55,5.4,cash,<NA>,None,other,90.61,265.92,181.22,84.70,31.85
4,13,1,3,3210,205,4,2025-09-03,179.13,32.4,credit card,<NA>,None,other,120.41,484.37,481.64,2.73,0.56
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9945,7937,10,87,4504,372,3,2025-04-16,109.51,33.7,cash,<NA>,None,other,44.57,217.82,133.71,84.11,38.61
9946,6711,11,63,2028,805,1,2025-10-16,51.18,26.7,credit card,loyalty tier discount applied,None,loyalty_tier,30.08,37.51,30.08,7.43,19.81
9947,8450,9,27,26,288,1,2025-04-10,48.36,5.2,credit card,loyalty tier discount applied,None,loyalty_tier,32.44,45.85,32.44,13.41,29.25
9948,637,10,39,2571,282,1,2025-11-23,115.75,28.0,credit card,promo code welcome15 applied at checkout,welcome15,promo_code,79.35,83.34,79.35,3.99,4.79


In [ ]:
!pip install -q  pyngrok plotly streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 55.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 66.9 MB/s eta 0:00:00


In [ ]:
%%writefile app.py

import streamlit as st
import plotly.express as px
import pandas as pd

st.set_page_config(
    page_title="Miredian Pharmacies",
    page_icon="💊",
    layout="wide"
)

st.title("Miredian Pharmacies")

st.write("Dashboard is running 🚀")

df = pd.DataFrame({
    "Month": ["Jan", "Feb", "Mar", "Apr", "May"],
    "Revenue": [12000, 15000, 13500, 18000, 21000]
})

fig = px.line(
    df,
    x="Month",
    y="Revenue",
    markers=True,
    title="Revenue Trend"
)

fig.update_layout(
    template="plotly_dark"
)

st.plotly_chart(
    fig,
    use_container_width=True
)

Overwriting app.py


In [ ]:
from pyngrok import ngrok

ngrok.set_auth_token("2vxWmOORNMRZzEPLGZDLOSGk2yf_6mjNXxHT5SDeyBUz6EFtc")

In [ ]:
from pyngrok import ngrok
import subprocess
import time

process = subprocess.Popen(
    [
        "streamlit",
        "run",
        "app.py",
        "--server.port",
        "8501",
        "--server.address",
        "0.0.0.0"
    ]
)

time.sleep(3)

public_url = ngrok.connect(8501)

print("Dashboard URL:")
print(public_url)

Dashboard URL:
NgrokTunnel: "https://a898-104-197-222-22.ngrok-free.app" -> "http://localhost:8501"


In [ ]:
%%writefile app.py

import os
import json
import time
import html
import math

import pandas as pd
import streamlit as st
import plotly.graph_objects as go
import plotly.express as px

from geopy.geocoders import Nominatim


# =========================================================
# PAGE CONFIG
# =========================================================

st.set_page_config(
    page_title="Miredian Pharmacies",
    page_icon="💊",
    layout="wide",
    initial_sidebar_state="expanded"
)


# =========================================================
# GLOBAL CSS
# =========================================================

st.markdown("""
<style>

.stApp {
    background-color: #0b0f19;
}

section[data-testid="stSidebar"] {
    background-color: #111827;
    border-right: 1px solid #1f2937;
}

section[data-testid="stSidebar"] > div {
    padding: 0.8rem;
}

section[data-testid="stSidebar"] [data-testid="column"] {
    padding: 0 !important;
}


/* ================= SIDEBAR ================= */

div.stButton {
    margin: 0;
    padding: 0;
}

div.stButton > button {
    width: 100%;
    min-height: 42px;
    border: none;
    border-radius: 9px;
    background: transparent;
    color: #9ca3af;
    text-align: left;
    padding: 10px 12px;
    margin: 3px 0;
    font-size: 13px;
    font-weight: 500;
    transition: background .18s ease, color .18s ease, transform .18s ease;
}

div.stButton > button:hover {
    background-color: #1f2937;
    color: #f9fafb;
    transform: translateX(2px);
}

div.stButton > button:focus {
    box-shadow: none;
    outline: none;
}

.brand-logo {
    width: 42px;
    height: 42px;
    border-radius: 12px;
    background: linear-gradient(135deg, #0ea5e9, #6366f1);
    display: flex;
    align-items: center;
    justify-content: center;
    color: white;
    font-size: 15px;
    font-weight: 800;
    margin-top: 2px;
}

.brand-name {
    color: #f9fafb;
    font-size: 15px;
    font-weight: 700;
    white-space: nowrap;
    line-height: 1.2;
    margin-top: 2px;
}

.brand-subtitle {
    color: #ffffff;
    font-size: 10px;
    white-space: nowrap;
    line-height: 1.2;
    margin-top: 4px;
}

.user-divider {
    height: 1px;
    background-color: #1f2937;
    margin-top: 28px;
    margin-bottom: 15px;
}

.user-avatar {
    width: 38px;
    height: 38px;
    border-radius: 50%;
    background: linear-gradient(135deg, #0ea5e9, #6366f1);
    display: flex;
    align-items: center;
    justify-content: center;
    color: white;
    font-size: 12px;
    font-weight: 700;
}

.user-name {
    color: #f9fafb;
    font-size: 12px;
    font-weight: 600;
    white-space: nowrap;
    line-height: 1.3;
    margin-top: 3px;
}

.user-role {
    color: #ffffff;
    font-size: 10px;
    white-space: nowrap;
    line-height: 1.3;
    margin-top: 3px;
}


/* ================= MAIN ================= */

.main-title {
    color: #f9fafb;
    font-size: 29px;
    font-weight: 750;
    margin-bottom: 2px;
}

.main-subtitle {
    color: #ffffff;
    font-size: 13px;
    margin-bottom: 20px;
}


/* ================= SECTION HEADERS ================= */

.section-header {
    display: flex;
    align-items: center;
    justify-content: center;
    gap: 15px;
    margin-top: 30px;
    margin-bottom: 19px;
}

.section-header::before,
.section-header::after {
    content: "";
    height: 1px;
    flex: 1;
    max-width: 190px;
    background: linear-gradient(
        90deg,
        transparent,
        #22d3ee,
        transparent
    );
    box-shadow: 0 0 8px rgba(34, 211, 238, 0.45);
}

.section-title {
    color: #67e8f9;
    font-size: 18px;
    font-weight: 750;
    letter-spacing: 0.35px;
    text-align: center;
    text-shadow:
        0 0 7px rgba(34, 211, 238, 0.65),
        0 0 18px rgba(34, 211, 238, 0.25);
    white-space: nowrap;
}


/* ================= KPI CARDS ================= */

.kpi-card {
    background: #111827;
    border: 1px solid #1f2937;
    border-radius: 14px;
    padding: 18px;
    min-height: 112px;

    display: flex;
    flex-direction: column;
    align-items: center;
    justify-content: center;

    text-align: center;

    transition:
        border-color .2s ease,
        transform .2s ease;
}

.kpi-card:hover {
    border-color: #334155;
    transform: translateY(-1px);
}

.kpi-label {
    color: #ffffff;
    font-size: 12px;
    font-weight: 600;
    margin-bottom: 7px;
}

.kpi-value {
    color: #f8fafc;
    font-size: 23px;
    font-weight: 750;
    letter-spacing: 0.2px;
    line-height: 1.2;
}

.kpi-sub {
    color: #ffffff;
    font-size: 11px;
    margin-top: 6px;
}


/* ================= CHART TITLES ================= */

.chart-title {
    color: #a5f3fc;
    font-size: 18px;
    font-weight: 750;
    margin-bottom: 5px;
    letter-spacing: 0.15px;
    text-shadow:
        0 0 7px rgba(34, 211, 238, 0.22);
}

.chart-subtitle {
    color: #ffffff;
    font-size: 12px;
    margin-bottom: 8px;
}


/* ================= SELECT DESCRIPTION ================= */

div[data-testid="stSelectbox"] label {
    color: #ffffff !important;
    font-size: 12px !important;
    font-weight: 500 !important;
}


/* ================= FILTERS ================= */

div[data-baseweb="select"] > div {
    background-color: #111827;
    border-color: #273449;
}


/* ================= DIVIDER ================= */

.dashboard-divider {
    height: 1px;
    background: linear-gradient(
        90deg,
        transparent,
        #1f2937,
        transparent
    );
    margin: 28px 0;
}

</style>
""", unsafe_allow_html=True)


# =========================================================
# HELPERS
# =========================================================

def section_header(title):

    safe_title = html.escape(str(title))

    st.html(
        f"""
        <div class="section-header">
            <div class="section-title">
                {safe_title}
            </div>
        </div>
        """
    )


def chart_title(title, subtitle=None):

    safe_title = html.escape(str(title))

    st.html(
        f"""
        <div class="chart-title">
            {safe_title}
        </div>
        """
    )

    if subtitle:

        safe_subtitle = html.escape(str(subtitle))

        st.html(
            f"""
            <div class="chart-subtitle">
                {safe_subtitle}
            </div>
            """
        )


def money(value):

    if pd.isna(value):
        value = 0

    return f"${value:,.2f}"


def number(value):

    if pd.isna(value):
        value = 0

    return f"{value:,.2f}"


def percent(value):

    if pd.isna(value):
        value = 0

    return f"{value:,.2f}%"


def dynamic_bar_positions(values, threshold=0.35):

    values = pd.to_numeric(
        pd.Series(values),
        errors="coerce"
    ).fillna(0)

    if len(values) == 0:
        return []

    max_value = values.max()

    if max_value <= 0:
        return ["outside"] * len(values)

    return [
        "inside"
        if value >= max_value * threshold
        else "outside"
        for value in values
    ]


def bar_text(values):

    return [
        f"{value:,.2f}"
        for value in pd.to_numeric(
            pd.Series(values),
            errors="coerce"
        ).fillna(0)
    ]


def standard_bar_layout(height=330):

    return dict(
        height=height,
        margin=dict(
            l=10,
            r=80,
            t=15,
            b=15
        ),
        paper_bgcolor="#111827",
        plot_bgcolor="#111827",
        font=dict(
            color="#e2e8f0",
            size=12
        ),
        xaxis=dict(
            showgrid=True,
            gridcolor="#243044",
            zeroline=False,
            tickfont=dict(
                color="#dbe4ee",
                size=11
            )
        ),
        yaxis=dict(
            showgrid=False,
            tickfont=dict(
                color="#dbe4ee",
                size=12
            )
        )
    )


# =========================================================
# GEOGRAPHIC HELPERS
# =========================================================

def haversine_distance(lat1, lon1, lat2, lon2):

    """
    Calculate distance between two geographic points in kilometers.
    """

    if any(
        pd.isna(value)
        for value in [
            lat1,
            lon1,
            lat2,
            lon2
        ]
    ):
        return float("nan")

    radius = 6371.0

    lat1 = math.radians(float(lat1))
    lon1 = math.radians(float(lon1))
    lat2 = math.radians(float(lat2))
    lon2 = math.radians(float(lon2))

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        math.sin(dlat / 2) ** 2
        +
        math.cos(lat1)
        * math.cos(lat2)
        * math.sin(dlon / 2) ** 2
    )

    c = 2 * math.atan2(
        math.sqrt(a),
        math.sqrt(1 - a)
    )

    return radius * c


def nearest_store_analysis(
    customer_geo,
    store_geo
):

    """
    For every customer city, find the geographically nearest
    store city using city-level coordinates.
    """

    if (
        customer_geo.empty
        or store_geo.empty
    ):
        return pd.DataFrame()

    store_points = store_geo[
        [
            "city",
            "lat",
            "lon",
            "Transactions",
            "Revenue",
            "Profit"
        ]
    ].dropna(
        subset=[
            "lat",
            "lon"
        ]
    ).copy()

    customer_points = customer_geo[
        [
            "city",
            "lat",
            "lon",
            "Transactions",
            "Revenue",
            "Profit"
        ]
    ].dropna(
        subset=[
            "lat",
            "lon"
        ]
    ).copy()

    results = []

    for _, customer in customer_points.iterrows():

        best_store = None
        best_distance = float("inf")

        for _, store in store_points.iterrows():

            if (
                customer["lat"] == store["lat"]
                and
                customer["lon"] == store["lon"]
            ):
                distance = 0.0

            else:

                distance = haversine_distance(
                    customer["lat"],
                    customer["lon"],
                    store["lat"],
                    store["lon"]
                )

            if distance < best_distance:

                best_distance = distance
                best_store = store

        if best_store is not None:

            results.append(
                {
                    "Customer City":
                        customer["city"],

                    "Customer Transactions":
                        customer["Transactions"],

                    "Customer Revenue":
                        customer["Revenue"],

                    "Customer Profit":
                        customer["Profit"],

                    "Nearest Store":
                        best_store["city"],

                    "Nearest Store Transactions":
                        best_store["Transactions"],

                    "Nearest Store Revenue":
                        best_store["Revenue"],

                    "Nearest Store Profit":
                        best_store["Profit"],

                    "Distance (km)":
                        best_distance,

                    "lat":
                        customer["lat"],

                    "lon":
                        customer["lon"]
                }
            )

    return pd.DataFrame(results)


def store_overlap_analysis(store_details):

    """
    Find pairs of store locations that are geographically close.
    This works at city-level when only city coordinates are available.
    """

    if len(store_details) < 2:

        return pd.DataFrame()

    records = []

    stores_list = (
        store_details
        .dropna(
            subset=[
                "lat",
                "lon"
            ]
        )
        .reset_index(drop=True)
    )

    for i in range(len(stores_list)):

        store_a = stores_list.iloc[i]

        for j in range(
            i + 1,
            len(stores_list)
        ):

            store_b = stores_list.iloc[j]

            distance = haversine_distance(
                store_a["lat"],
                store_a["lon"],
                store_b["lat"],
                store_b["lon"]
            )

            records.append(
                {
                    "Store A":
                        store_a["store_name"],

                    "City A":
                        store_a["store_city"],

                    "Transactions A":
                        store_a["Transactions"],

                    "Revenue A":
                        store_a["Revenue"],

                    "Store B":
                        store_b["store_name"],

                    "City B":
                        store_b["store_city"],

                    "Transactions B":
                        store_b["Transactions"],

                    "Revenue B":
                        store_b["Revenue"],

                    "Distance (km)":
                        distance,

                    "Combined Transactions":
                        (
                            store_a["Transactions"]
                            +
                            store_b["Transactions"]
                        )
                }
            )

    return pd.DataFrame(records)


# =========================================================
# LOAD DATA
# =========================================================

DATA_PATH = "dashboard_data"

required_files = [
    "categories",
    "customers",
    "employees",
    "prescriptions",
    "products",
    "stock",
    "suppliers",
    "stores",
    "sales"
]

data = {}
missing_files = []

for name in required_files:

    path = os.path.join(
        DATA_PATH,
        f"{name}.parquet"
    )

    if os.path.exists(path):

        data[name] = pd.read_parquet(path)

    else:

        missing_files.append(name)


if missing_files:

    st.error(
        "Missing dashboard data files: "
        + ", ".join(missing_files)
    )

    st.stop()


categories = data["categories"]
customers = data["customers"]
employees = data["employees"]
prescriptions = data["prescriptions"]
products = data["products"]
stock = data["stock"]
suppliers = data["suppliers"]
stores = data["stores"]
sales = data["sales"].copy()


# =========================================================
# PREPARE SALES
# =========================================================

sales["quantity"] = pd.to_numeric(
    sales["quantity"],
    errors="coerce"
).fillna(0)

sales["unit_price"] = pd.to_numeric(
    sales["unit_price"],
    errors="coerce"
).fillna(0)

sales["discount_pct"] = pd.to_numeric(
    sales["discount_pct"],
    errors="coerce"
).fillna(0)

sales["cost_price"] = pd.to_numeric(
    sales["cost_price"],
    errors="coerce"
).fillna(0)

sales["total_amount"] = pd.to_numeric(
    sales["total_amount"],
    errors="coerce"
)

sales["total_cost"] = pd.to_numeric(
    sales["total_cost"],
    errors="coerce"
)

sales["profit"] = pd.to_numeric(
    sales["profit"],
    errors="coerce"
)

sales["margin"] = pd.to_numeric(
    sales["margin"],
    errors="coerce"
)

sales["sale_date"] = pd.to_datetime(
    sales["sale_date"],
    errors="coerce"
)


# =========================================================
# FINANCIAL FALLBACKS
# =========================================================

calculated_revenue = (
    sales["quantity"]
    * sales["unit_price"]
    * (1 - sales["discount_pct"] / 100)
)

calculated_cost = (
    sales["quantity"]
    * sales["cost_price"]
)


sales["total_amount"] = (
    sales["total_amount"]
    .fillna(calculated_revenue)
    .round(2)
)

sales["total_cost"] = (
    sales["total_cost"]
    .fillna(calculated_cost)
    .round(2)
)

sales["profit"] = (
    sales["profit"]
    .fillna(
        sales["total_amount"]
        - sales["total_cost"]
    )
    .round(2)
)


# =========================================================
# DISCOUNT
# =========================================================

sales["gross_amount"] = (
    sales["quantity"]
    * sales["unit_price"]
).round(2)

sales["discount_amount"] = (
    sales["gross_amount"]
    - sales["total_amount"]
).clip(
    lower=0
).round(2)


# =========================================================
# PRODUCT / CATEGORY
# =========================================================

product_lookup = products[
    [
        "product_id",
        "category_id",
        "product_name"
    ]
].copy()


sales = sales.merge(
    product_lookup,
    on="product_id",
    how="left",
    suffixes=("", "_product")
)


category_lookup = categories[
    [
        "category_id",
        "category_name"
    ]
].copy()


sales = sales.merge(
    category_lookup,
    on="category_id",
    how="left"
)


sales["category_name"] = (
    sales["category_name"]
    .fillna("Unknown")
)


# =========================================================
# STORE
# =========================================================

store_lookup = stores[
    [
        "store_id",
        "store_name",
        "city",
        "region"
    ]
].copy()


store_lookup = store_lookup.rename(
    columns={
        "city": "store_city"
    }
)


sales = sales.merge(
    store_lookup,
    on="store_id",
    how="left"
)


sales["store_name"] = (
    sales["store_name"]
    .fillna("Unknown")
)

sales["store_city"] = (
    sales["store_city"]
    .fillna("Unknown")
)


# =========================================================
# CUSTOMER / LOYALTY
# =========================================================

customer_lookup = customers[
    [
        "customer_id",
        "city",
        "loyalty_tier"
    ]
].copy()


customer_lookup = customer_lookup.rename(
    columns={
        "city": "customer_city"
    }
)


sales = sales.merge(
    customer_lookup,
    on="customer_id",
    how="left"
)


sales["customer_city"] = (
    sales["customer_city"]
    .fillna("Unknown")
)

sales["loyalty_tier"] = (
    sales["loyalty_tier"]
    .fillna("No Loyalty")
)


# =========================================================
# GEOLOCATION
# =========================================================

GEO_CACHE_FILE = os.path.join(
    DATA_PATH,
    "city_coordinates.json"
)


def load_geo_cache():

    if os.path.exists(GEO_CACHE_FILE):

        try:

            with open(
                GEO_CACHE_FILE,
                "r",
                encoding="utf-8"
            ) as f:

                return json.load(f)

        except Exception:

            return {}

    return {}


def save_geo_cache(cache):

    try:

        with open(
            GEO_CACHE_FILE,
            "w",
            encoding="utf-8"
        ) as f:

            json.dump(
                cache,
                f,
                ensure_ascii=False,
                indent=2
            )

    except Exception:

        pass


@st.cache_data(ttl=86400)
def geocode_cities(city_list):

    cache = load_geo_cache()

    geolocator = Nominatim(
        user_agent="miredian-pharmacies-dashboard"
    )

    result = {}

    for city in city_list:

        city_key = str(city).strip()

        if not city_key or city_key == "Unknown":
            continue

        if city_key in cache:

            result[city_key] = cache[city_key]
            continue

        try:

            location = geolocator.geocode(
                city_key,
                timeout=10
            )

            if location:

                coords = {
                    "lat": float(location.latitude),
                    "lon": float(location.longitude)
                }

                cache[city_key] = coords
                result[city_key] = coords

                save_geo_cache(cache)

                time.sleep(1)

        except Exception:

            continue

    return result


# =========================================================
# SIDEBAR
# =========================================================

with st.sidebar:

    logo_col, brand_col = st.columns(
        [0.22, 0.78],
        gap="small"
    )

    with logo_col:

        st.markdown(
            '<div class="brand-logo">MP</div>',
            unsafe_allow_html=True
        )

    with brand_col:

        st.markdown(
            """
            <div class="brand-name">
                Miredian Pharmacies
            </div>
            <div class="brand-subtitle">
                Management Dashboard
            </div>
            """,
            unsafe_allow_html=True
        )


    st.markdown(
        '<div class="user-divider"></div>',
        unsafe_allow_html=True
    )


    pages = [
        ("📊", "Overview"),
        ("💰", "Sales"),
        ("👥", "Customer Loyalty"),
        ("📦", "Inventory"),
        ("👨‍💼", "Employees"),
        ("🚚", "Suppliers")
    ]


    if "current_page" not in st.session_state:

        st.session_state.current_page = "Overview"


    for icon, page_name in pages:

        if st.button(
            f"{icon}   {page_name}",
            key=f"nav_{page_name}"
        ):

            st.session_state.current_page = page_name
            st.rerun()


    st.markdown(
        '<div class="user-divider"></div>',
        unsafe_allow_html=True
    )


    avatar_col, user_col = st.columns(
        [0.22, 0.78],
        gap="small"
    )


    with avatar_col:

        st.markdown(
            '<div class="user-avatar">OO</div>',
            unsafe_allow_html=True
        )


    with user_col:

        st.markdown(
            """
            <div class="user-name">
                Omar Osama
            </div>
            <div class="user-role">
                System Admin
            </div>
            """,
            unsafe_allow_html=True
        )


page = st.session_state.current_page


# =========================================================
# OVERVIEW
# =========================================================

if page == "Overview":

    st.markdown(
        '<div class="main-title">Overview</div>',
        unsafe_allow_html=True
    )

    st.markdown(
        '<div class="main-subtitle">'
        'Miredian Pharmacies · Business Performance'
        '</div>',
        unsafe_allow_html=True
    )


    section_header(
        "Business Performance"
    )


    col1, col2, col3, col4 = st.columns(4)


    overview_values = [
        (
            "Revenue",
            "$284,600.00",
            "+12.40% vs previous period"
        ),
        (
            "Profit",
            "$71,800.00",
            "+9.80% vs previous period"
        ),
        (
            "Profit Margin",
            "25.20%",
            "Weighted margin"
        ),
        (
            "Customers",
            "1,248.00",
            "Active customers"
        )
    ]


    for col, data_point in zip(
        [col1, col2, col3, col4],
        overview_values
    ):

        with col:

            st.markdown(
                f"""
                <div class="kpi-card">
                    <div class="kpi-label">
                        {data_point[0]}
                    </div>
                    <div class="kpi-value">
                        {data_point[1]}
                    </div>
                    <div class="kpi-sub">
                        {data_point[2]}
                    </div>
                </div>
                """,
                unsafe_allow_html=True
            )


    left, right = st.columns(
        [2, 1]
    )


    with left:

        chart_title(
            "Revenue & Profit Trend",
            "Monthly business performance"
        )


        months = [
            "Jan", "Feb", "Mar", "Apr",
            "May", "Jun", "Jul", "Aug"
        ]

        revenue = [
            28000, 31000, 34000, 32000,
            39000, 41000, 38000, 44000
        ]

        profit = [
            7000, 8000, 9000, 8000,
            10000, 11000, 10000, 12000
        ]


        fig = go.Figure()


        fig.add_trace(
            go.Scatter(
                x=months,
                y=revenue,
                mode="lines+markers",
                name="Revenue",
                line=dict(width=3),
                marker=dict(size=7)
            )
        )


        fig.add_trace(
            go.Scatter(
                x=months,
                y=profit,
                mode="lines+markers",
                name="Profit",
                line=dict(width=3),
                marker=dict(size=7)
            )
        )


        fig.update_layout(
            height=310,
            margin=dict(
                l=10,
                r=10,
                t=10,
                b=10
            ),
            paper_bgcolor="#111827",
            plot_bgcolor="#111827",
            font=dict(
                color="#e2e8f0"
            ),
            legend=dict(
                orientation="h"
            ),
            xaxis=dict(
                showgrid=False
            ),
            yaxis=dict(
                gridcolor="#243044"
            )
        )


        st.plotly_chart(
            fig,
            use_container_width=True,
            config={
                "displayModeBar": False
            }
        )


    with right:

        chart_title(
            "Revenue by Category",
            "Revenue contribution across product categories"
        )


        category_values = [
            42,
            27,
            19,
            12
        ]


        category_names = [
            "Medicine",
            "Personal Care",
            "Supplements",
            "Baby Care"
        ]


        positions = dynamic_bar_positions(
            category_values
        )


        fig = go.Figure(
            go.Bar(
                x=category_values,
                y=category_names,
                orientation="h",
                text=bar_text(
                    category_values
                ),
                textposition=positions,
                textfont=dict(
                    size=12,
                    color="#ffffff"
                ),
                cliponaxis=False
            )
        )


        fig.update_layout(
            **standard_bar_layout(310)
        )


        st.plotly_chart(
            fig,
            use_container_width=True,
            config={
                "displayModeBar": False
            }
        )


# =========================================================
# SALES
# =========================================================

elif page == "Sales":

    st.markdown(
        '<div class="main-title">Sales</div>',
        unsafe_allow_html=True
    )

    st.markdown(
        '<div class="main-subtitle">'
        'Sales Performance · Revenue · Profitability · Discounts'
        '</div>',
        unsafe_allow_html=True
    )


    # =====================================================
    # FILTERS
    # =====================================================

    filter_col1, filter_col2, filter_col3 = st.columns(3)


    with filter_col1:

        store_options = [
            "All Stores"
        ] + sorted(
            sales["store_name"]
            .dropna()
            .astype(str)
            .unique()
            .tolist()
        )


        selected_store = st.selectbox(
            "Store",
            store_options
        )


    with filter_col2:

        payment_options = [
            "All Payment Methods"
        ] + sorted(
            sales["payment_method"]
            .dropna()
            .astype(str)
            .unique()
            .tolist()
        )


        selected_payment = st.selectbox(
            "Payment Method",
            payment_options
        )


    with filter_col3:

        category_options = [
            "All Categories"
        ] + sorted(
            sales["category_name"]
            .dropna()
            .astype(str)
            .unique()
            .tolist()
        )


        selected_category = st.selectbox(
            "Category",
            category_options
        )


    filtered_sales = sales.copy()


    if selected_store != "All Stores":

        filtered_sales = filtered_sales[
            filtered_sales["store_name"]
            == selected_store
        ]


    if selected_payment != "All Payment Methods":

        filtered_sales = filtered_sales[
            filtered_sales["payment_method"]
            .astype(str)
            == selected_payment
        ]


    if selected_category != "All Categories":

        filtered_sales = filtered_sales[
            filtered_sales["category_name"]
            == selected_category
        ]


    # =====================================================
    # FINANCIAL OVERVIEW
    # =====================================================

    section_header(
        "Financial Overview"
    )


    total_revenue = (
        filtered_sales["total_amount"]
        .sum()
    )


    total_profit = (
        filtered_sales["profit"]
        .sum()
    )


    total_cost = (
        filtered_sales["total_cost"]
        .sum()
    )


    profit_margin = (
        total_profit
        / total_revenue
        * 100
        if total_revenue != 0
        else 0
    )


    transaction_count = (
        filtered_sales["sale_id"]
        .nunique()
    )


    c1, c2, c3, c4 = st.columns(4)


    financial_cards = [
        (
            "Total Revenue",
            money(total_revenue),
            f"{transaction_count:,.2f} transactions"
        ),
        (
            "Total Profit",
            money(total_profit),
            "Revenue − Cost"
        ),
        (
            "Profit Margin",
            percent(profit_margin),
            "Weighted margin"
        ),
        (
            "Total Cost",
            money(total_cost),
            "Cost of goods sold"
        )
    ]


    for col, card in zip(
        [c1, c2, c3, c4],
        financial_cards
    ):

        with col:

            st.markdown(
                f"""
                <div class="kpi-card">
                    <div class="kpi-label">
                        {card[0]}
                    </div>
                    <div class="kpi-value">
                        {card[1]}
                    </div>
                    <div class="kpi-sub">
                        {card[2]}
                    </div>
                </div>
                """,
                unsafe_allow_html=True
            )


    # =====================================================
    # ORDERS
    # =====================================================

    section_header(
        "Order Performance"
    )


    aov = (
        total_revenue
        / transaction_count
        if transaction_count != 0
        else 0
    )


    c1, c2 = st.columns(2)


    with c1:

        st.markdown(
            f"""
            <div class="kpi-card">
                <div class="kpi-label">
                    Average Order Value
                </div>
                <div class="kpi-value">
                    {money(aov)}
                </div>
                <div class="kpi-sub">
                    Average revenue per order
                </div>
            </div>
            """,
            unsafe_allow_html=True
        )


    with c2:

        st.markdown(
            f"""
            <div class="kpi-card">
                <div class="kpi-label">
                    Total Orders
                </div>
                <div class="kpi-value">
                    {transaction_count:,.2f}
                </div>
                <div class="kpi-sub">
                    Unique sales transactions
                </div>
            </div>
            """,
            unsafe_allow_html=True
        )


    # =====================================================
    # DISCOUNTS
    # =====================================================

    section_header(
        "Discount Performance"
    )


    total_discount = (
        filtered_sales["discount_amount"]
        .sum()
    )


    profit_before_discount = (
        filtered_sales["gross_amount"].sum()
        - filtered_sales["total_cost"].sum()
    )


    discount_pct_of_profit = (
        total_discount
        / profit_before_discount
        * 100
        if profit_before_discount != 0
        else 0
    )


    avg_discount = (
        filtered_sales["discount_pct"].mean()
        if len(filtered_sales) > 0
        else 0
    )


    c1, c2, c3 = st.columns(3)


    discount_cards = [
        (
            "Total Discount Amount",
            money(total_discount),
            "Absolute discount value"
        ),
        (
            "Discount % of Profit",
            percent(discount_pct_of_profit),
            "Discount ÷ profit before discount"
        ),
        (
            "Average Discount",
            percent(avg_discount),
            "Average discount rate"
        )
    ]


    for col, card in zip(
        [c1, c2, c3],
        discount_cards
    ):

        with col:

            st.markdown(
                f"""
                <div class="kpi-card">
                    <div class="kpi-label">
                        {card[0]}
                    </div>
                    <div class="kpi-value">
                        {card[1]}
                    </div>
                    <div class="kpi-sub">
                        {card[2]}
                    </div>
                </div>
                """,
                unsafe_allow_html=True
            )


    # =====================================================
    # VOLUME
    # =====================================================

    section_header(
        "Volume Metrics"
    )


    total_quantity = (
        filtered_sales["quantity"]
        .sum()
    )


    st.markdown(
        f"""
        <div class="kpi-card">
            <div class="kpi-label">
                Total Quantities Sold
            </div>
            <div class="kpi-value">
                {number(total_quantity)}
            </div>
            <div class="kpi-sub">
                Total units sold
            </div>
        </div>
        """,
        unsafe_allow_html=True
    )


    # =====================================================
    # CATEGORY / CITY
    # =====================================================

    section_header(
        "Category & City Performance"
    )


    left, right = st.columns(2)


    with left:

        chart_title(
            "Category Performance",
            "Compare categories by the selected metric"
        )


        category_metric = st.selectbox(
            "Select the metric to compare categories",
            [
                "Revenue",
                "Profit",
                "Profit Margin",
                "Total Cost"
            ],
            key="category_metric"
        )


        category_group = (
            filtered_sales
            .groupby(
                "category_name",
                dropna=False
            )
            .agg(
                Revenue=(
                    "total_amount",
                    "sum"
                ),
                Profit=(
                    "profit",
                    "sum"
                ),
                Total_Cost=(
                    "total_cost",
                    "sum"
                )
            )
            .reset_index()
        )


        category_group["Profit Margin"] = (
            category_group["Profit"]
            / category_group["Revenue"]
            * 100
        ).where(
            category_group["Revenue"] != 0
        ).fillna(0)


        metric_column_map = {
            "Revenue": "Revenue",
            "Profit": "Profit",
            "Profit Margin": "Profit Margin",
            "Total Cost": "Total_Cost"
        }


        selected_column = metric_column_map[
            category_metric
        ]


        category_group = (
            category_group
            .sort_values(
                selected_column,
                ascending=True
            )
        )


        positions = dynamic_bar_positions(
            category_group[selected_column]
        )


        fig = go.Figure(
            go.Bar(
                x=category_group[selected_column],
                y=category_group["category_name"],
                orientation="h",
                text=bar_text(
                    category_group[selected_column]
                ),
                textposition=positions,
                textfont=dict(
                    size=12,
                    color="#ffffff"
                ),
                cliponaxis=False
            )
        )


        fig.update_layout(
            **standard_bar_layout(350)
        )


        st.plotly_chart(
            fig,
            use_container_width=True,
            config={
                "displayModeBar": False
            }
        )


    with right:

        chart_title(
            "City Performance",
            "Compare store cities by the selected metric"
        )


        city_metric = st.selectbox(
            "Select the metric to compare cities",
            [
                "Transaction Count",
                "Revenue",
                "Profit",
                "Profit Margin",
                "Cost"
            ],
            key="city_metric"
        )


        city_group = (
            filtered_sales
            .groupby(
                "store_city",
                dropna=False
            )
            .agg(
                Transaction_Count=(
                    "sale_id",
                    "nunique"
                ),
                Revenue=(
                    "total_amount",
                    "sum"
                ),
                Profit=(
                    "profit",
                    "sum"
                ),
                Cost=(
                    "total_cost",
                    "sum"
                )
            )
            .reset_index()
        )


        city_group["Profit Margin"] = (
            city_group["Profit"]
            / city_group["Revenue"]
            * 100
        ).where(
            city_group["Revenue"] != 0
        ).fillna(0)


        city_column_map = {
            "Transaction Count":
                "Transaction_Count",
            "Revenue":
                "Revenue",
            "Profit":
                "Profit",
            "Profit Margin":
                "Profit Margin",
            "Cost":
                "Cost"
        }


        city_column = city_column_map[
            city_metric
        ]


        city_group = (
            city_group
            .sort_values(
                city_column,
                ascending=True
            )
        )


        positions = dynamic_bar_positions(
            city_group[city_column]
        )


        fig = go.Figure(
            go.Bar(
                x=city_group[city_column],
                y=city_group["store_city"],
                orientation="h",
                text=bar_text(
                    city_group[city_column]
                ),
                textposition=positions,
                textfont=dict(
                    size=12,
                    color="#ffffff"
                ),
                cliponaxis=False
            )
        )


        fig.update_layout(
            **standard_bar_layout(350)
        )


        st.plotly_chart(
            fig,
            use_container_width=True,
            config={
                "displayModeBar": False
            }
        )


    # =====================================================
    # PROMO + LOYALTY
    # =====================================================

    section_header(
        "Promo Codes & Customer Loyalty"
    )


    left, right = st.columns(2)


    with left:

        chart_title(
            "Promo Code Analytics",
            "Compare promo code performance by the selected metric"
        )


        promo_metric = st.selectbox(
            "Select the metric to compare promo codes",
            [
                "Transaction Count",
                "Revenue",
                "Profit",
                "Profit Margin"
            ],
            key="promo_metric"
        )


        promo_sales = filtered_sales.copy()


        promo_sales["promo_display"] = (
            promo_sales["promo_code"]
            .fillna("")
            .astype(str)
            .str.strip()
        )


        promo_sales.loc[
            promo_sales["promo_display"] == "",
            "promo_display"
        ] = "No Promo Code"


        promo_group = (
            promo_sales
            .groupby(
                "promo_display"
            )
            .agg(
                Transaction_Count=(
                    "sale_id",
                    "nunique"
                ),
                Revenue=(
                    "total_amount",
                    "sum"
                ),
                Profit=(
                    "profit",
                    "sum"
                )
            )
            .reset_index()
        )


        promo_group["Profit Margin"] = (
            promo_group["Profit"]
            / promo_group["Revenue"]
            * 100
        ).where(
            promo_group["Revenue"] != 0
        ).fillna(0)


        promo_column_map = {
            "Transaction Count":
                "Transaction_Count",
            "Revenue":
                "Revenue",
            "Profit":
                "Profit",
            "Profit Margin":
                "Profit Margin"
        }


        promo_column = promo_column_map[
            promo_metric
        ]


        promo_group = (
            promo_group
            .sort_values(
                promo_column,
                ascending=True
            )
            .tail(10)
        )


        positions = dynamic_bar_positions(
            promo_group[promo_column]
        )


        fig = go.Figure(
            go.Bar(
                x=promo_group[promo_column],
                y=promo_group["promo_display"],
                orientation="h",
                text=bar_text(
                    promo_group[promo_column]
                ),
                textposition=positions,
                textfont=dict(
                    size=12,
                    color="#ffffff"
                ),
                cliponaxis=False
            )
        )


        fig.update_layout(
            **standard_bar_layout(330)
        )


        st.plotly_chart(
            fig,
            use_container_width=True,
            config={
                "displayModeBar": False
            }
        )


    with right:

        chart_title(
            "Customer Loyalty Performance",
            "Compare loyalty types by the selected metric"
        )


        loyalty_metric = st.selectbox(
            "Select the metric to compare loyalty types",
            [
                "Transaction Count",
                "Revenue",
                "Profit",
                "Profit Margin"
            ],
            key="loyalty_metric"
        )


        loyalty_group = (
            filtered_sales
            .groupby(
                "loyalty_tier"
            )
            .agg(
                Transactions=(
                    "sale_id",
                    "nunique"
                ),
                Revenue=(
                    "total_amount",
                    "sum"
                ),
                Profit=(
                    "profit",
                    "sum"
                )
            )
            .reset_index()
        )


        loyalty_group["Profit Margin"] = (
            loyalty_group["Profit"]
            / loyalty_group["Revenue"]
            * 100
        ).where(
            loyalty_group["Revenue"] != 0
        ).fillna(0)


        loyalty_column_map = {
            "Transaction Count":
                "Transactions",
            "Revenue":
                "Revenue",
            "Profit":
                "Profit",
            "Profit Margin":
                "Profit Margin"
        }


        loyalty_column = loyalty_column_map[
            loyalty_metric
        ]


        loyalty_group = (
            loyalty_group
            .sort_values(
                loyalty_column,
                ascending=True
            )
        )


        positions = dynamic_bar_positions(
            loyalty_group[loyalty_column]
        )


        fig = go.Figure(
            go.Bar(
                x=loyalty_group[loyalty_column],
                y=loyalty_group["loyalty_tier"],
                orientation="h",
                text=bar_text(
                    loyalty_group[loyalty_column]
                ),
                textposition=positions,
                textfont=dict(
                    size=12,
                    color="#ffffff"
                ),
                cliponaxis=False
            )
        )


        fig.update_layout(
            **standard_bar_layout(330)
        )


        st.plotly_chart(
            fig,
            use_container_width=True,
            config={
                "displayModeBar": False
            }
        )


    # =====================================================
    # PAYMENT METHODS
    # =====================================================

    section_header(
        "Payment Methods"
    )


    payment_metric = st.selectbox(
        "Select the metric to compare payment methods",
        [
            "Transaction Count",
            "Total Revenue"
        ],
        key="payment_metric"
    )


    payment_group = (
        filtered_sales
        .groupby(
            "payment_method",
            dropna=False
        )
        .agg(
            Transaction_Count=(
                "sale_id",
                "nunique"
            ),
            Total_Revenue=(
                "total_amount",
                "sum"
            )
        )
        .reset_index()
    )


    payment_group["payment_method"] = (
        payment_group["payment_method"]
        .fillna("Unknown")
    )


    payment_column = (
        "Transaction_Count"
        if payment_metric == "Transaction Count"
        else "Total_Revenue"
    )


    payment_group = (
        payment_group
        .sort_values(
            payment_column,
            ascending=True
        )
    )


    positions = dynamic_bar_positions(
        payment_group[payment_column]
    )


    fig = go.Figure(
        go.Bar(
            x=payment_group[payment_column],
            y=payment_group["payment_method"],
            orientation="h",
            text=bar_text(
                payment_group[payment_column]
            ),
            textposition=positions,
            textfont=dict(
                size=12,
                color="#ffffff"
            ),
            cliponaxis=False
        )
    )


    fig.update_layout(
        **standard_bar_layout(330)
    )


    st.plotly_chart(
        fig,
        use_container_width=True,
        config={
            "displayModeBar": False
        }
    )


    # =====================================================
    # DISCOUNT STATUS
    # =====================================================

    section_header(
        "Discount Status & Product Volume"
    )


    left, right = st.columns(2)


    with left:

        chart_title(
            "Discount Status",
            "Compare orders with discounts against orders marked No Discount"
        )


        discount_sales = (
            filtered_sales.copy()
        )


        discount_sales["discount_status"] = (
            discount_sales["discount_pct"] > 0
        ).map(
            {
                True: "With Discount",
                False: "No Discount"
            }
        )


        discount_group = (
            discount_sales
            .groupby(
                "discount_status"
            )
            .agg(
                Orders=(
                    "sale_id",
                    "nunique"
                )
            )
            .reset_index()
        )


        positions = dynamic_bar_positions(
            discount_group["Orders"]
        )


        fig = go.Figure(
            go.Bar(
                x=discount_group["discount_status"],
                y=discount_group["Orders"],
                text=bar_text(
                    discount_group["Orders"]
                ),
                textposition=positions,
                textfont=dict(
                    size=13,
                    color="#ffffff"
                )
            )
        )


        fig.update_layout(
            height=320,
            margin=dict(
                l=10,
                r=40,
                t=15,
                b=15
            ),
            paper_bgcolor="#111827",
            plot_bgcolor="#111827",
            font=dict(
                color="#e2e8f0"
            ),
            xaxis=dict(
                showgrid=False
            ),
            yaxis=dict(
                gridcolor="#243044"
            )
        )


        st.plotly_chart(
            fig,
            use_container_width=True,
            config={
                "displayModeBar": False
            }
        )


    with right:

        chart_title(
            "Quantity by Category",
            "Compare total units sold across product categories"
        )


        quantity_group = (
            filtered_sales
            .groupby(
                "category_name"
            )["quantity"]
            .sum()
            .reset_index()
            .sort_values(
                "quantity",
                ascending=True
            )
        )


        positions = dynamic_bar_positions(
            quantity_group["quantity"]
        )


        fig = go.Figure(
            go.Bar(
                x=quantity_group["quantity"],
                y=quantity_group["category_name"],
                orientation="h",
                text=bar_text(
                    quantity_group["quantity"]
                ),
                textposition=positions,
                textfont=dict(
                    size=12,
                    color="#ffffff"
                ),
                cliponaxis=False
            )
        )


        fig.update_layout(
            **standard_bar_layout(320)
        )


        st.plotly_chart(
            fig,
            use_container_width=True,
            config={
                "displayModeBar": False
            }
        )


    # =====================================================
    # TIME TREND
    # =====================================================

    section_header(
        "Time-Based Sales Trend"
    )


    trend_col1, trend_col2 = st.columns(2)


    with trend_col1:

        time_granularity = st.selectbox(
            "Time Granularity",
            [
                "Years",
                "Quarters",
                "Months"
            ],
            key="time_granularity"
        )


    with trend_col2:

        trend_metric = st.selectbox(
            "Select the metric to compare over time",
            [
                "Revenue",
                "Profit",
                "Profit Margin"
            ],
            key="trend_metric"
        )


    trend_sales = filtered_sales[
        filtered_sales["sale_date"]
        .notna()
    ].copy()


    if len(trend_sales) > 0:

        if time_granularity == "Years":

            trend_sales["period"] = (
                trend_sales["sale_date"]
                .dt.year
                .astype(str)
            )

        elif time_granularity == "Quarters":

            trend_sales["period"] = (
                trend_sales["sale_date"]
                .dt.to_period("Q")
                .astype(str)
            )

        else:

            trend_sales["period"] = (
                trend_sales["sale_date"]
                .dt.to_period("M")
                .astype(str)
            )


        trend_group = (
            trend_sales
            .groupby("period")
            .agg(
                Revenue=(
                    "total_amount",
                    "sum"
                ),
                Profit=(
                    "profit",
                    "sum"
                )
            )
            .reset_index()
        )


        trend_group["Profit Margin"] = (
            trend_group["Profit"]
            / trend_group["Revenue"]
            * 100
        ).where(
            trend_group["Revenue"] != 0
        ).fillna(0)


        chart_title(
            f"{trend_metric} Trend",
            f"Compare {trend_metric.lower()} across {time_granularity.lower()}"
        )


        fig = go.Figure(
            go.Scatter(
                x=trend_group["period"],
                y=trend_group[trend_metric],
                mode="lines+markers",
                name=trend_metric,
                line=dict(
                    width=3
                ),
                marker=dict(
                    size=8
                ),
                hovertemplate=(
                    "%{x}"
                    "<br>"
                    + trend_metric
                    + ": %{y:,.2f}"
                    + "<extra></extra>"
                )
            )
        )


        fig.update_layout(
            height=380,
            margin=dict(
                l=10,
                r=20,
                t=15,
                b=15
            ),
            paper_bgcolor="#111827",
            plot_bgcolor="#111827",
            font=dict(
                color="#e2e8f0"
            ),
            xaxis=dict(
                showgrid=False,
                rangeslider=dict(
                    visible=False
                ),
                fixedrange=False
            ),
            yaxis=dict(
                gridcolor="#243044",
                fixedrange=False
            ),
            dragmode="zoom"
        )


        st.plotly_chart(
            fig,
            use_container_width=True,
            config={
                "displayModeBar": True,
                "scrollZoom": True,
                "displaylogo": False,
                "modeBarButtonsToAdd": [
                    "zoomIn2d",
                    "zoomOut2d",
                    "resetScale2d"
                ]
            }
        )


        missing_dates = (
            filtered_sales["sale_date"]
            .isna()
            .sum()
        )


        if missing_dates > 0:

            st.caption(
                f"{missing_dates:,.2f} records have no valid "
                "sale date and are excluded from the time trend."
            )


    else:

        st.info(
            "No valid sale dates are available "
            "for the selected filters."
        )


    # =====================================================
    # MARKET COVERAGE MAP
    # =====================================================

    section_header(
        "Market Coverage Map"
    )


    chart_title(
        "Customer Demand vs Existing Stores",
        "Compare where customers are located against existing pharmacy locations"
    )


    map_col1, map_col2 = st.columns(2)


    with map_col1:

        map_metric = st.selectbox(
            "Size customer demand by",
            [
                "Transaction Count",
                "Revenue",
                "Profit"
            ],
            key="market_map_metric"
        )


    with map_col2:

        demand_view = st.selectbox(
            "Customer demand view",
            [
                "All Customer Cities",
                "High Demand Cities"
            ],
            key="market_demand_view"
        )


    # =====================================================
    # STORE CITY AGGREGATION
    # =====================================================

    store_geo = (
        filtered_sales
        .groupby(
            "store_city",
            dropna=False
        )
        .agg(
            Transactions=(
                "sale_id",
                "nunique"
            ),
            Revenue=(
                "total_amount",
                "sum"
            ),
            Profit=(
                "profit",
                "sum"
            )
        )
        .reset_index()
        .rename(
            columns={
                "store_city": "city"
            }
        )
    )


    store_geo["Profit Margin"] = (
        store_geo["Profit"]
        / store_geo["Revenue"]
        * 100
    ).where(
        store_geo["Revenue"] != 0
    ).fillna(0)


    # =====================================================
    # CUSTOMER CITY AGGREGATION
    # =====================================================

    customer_geo = (
        filtered_sales
        .groupby(
            "customer_city",
            dropna=False
        )
        .agg(
            Transactions=(
                "sale_id",
                "nunique"
            ),
            Revenue=(
                "total_amount",
                "sum"
            ),
            Profit=(
                "profit",
                "sum"
            )
        )
        .reset_index()
        .rename(
            columns={
                "customer_city": "city"
            }
        )
    )


    customer_geo["Profit Margin"] = (
        customer_geo["Profit"]
        / customer_geo["Revenue"]
        * 100
    ).where(
        customer_geo["Revenue"] != 0
    ).fillna(0)


    # =====================================================
    # HIGH DEMAND FILTER
    # =====================================================

    if (
        demand_view == "High Demand Cities"
        and len(customer_geo) > 0
    ):

        demand_threshold = (
            customer_geo["Transactions"]
            .quantile(0.75)
        )

        customer_geo = customer_geo[
            customer_geo["Transactions"]
            >= demand_threshold
        ].copy()


    # =====================================================
    # COMBINED CITY LIST
    # =====================================================

    all_map_cities = sorted(
        set(
            store_geo["city"]
            .dropna()
            .astype(str)
            .str.strip()
            .tolist()
        )
        |
        set(
            customer_geo["city"]
            .dropna()
            .astype(str)
            .str.strip()
            .tolist()
        )
    )


    coordinates = geocode_cities(
        all_map_cities
    )


    # =====================================================
    # APPLY COORDINATES
    # =====================================================

    store_geo["lat"] = (
        store_geo["city"]
        .map(
            lambda x:
            coordinates.get(
                str(x).strip(),
                {}
            ).get("lat")
        )
    )


    store_geo["lon"] = (
        store_geo["city"]
        .map(
            lambda x:
            coordinates.get(
                str(x).strip(),
                {}
            ).get("lon")
        )
    )


    store_geo = store_geo.dropna(
        subset=[
            "lat",
            "lon"
        ]
    ).copy()


    customer_geo["lat"] = (
        customer_geo["city"]
        .map(
            lambda x:
            coordinates.get(
                str(x).strip(),
                {}
            ).get("lat")
        )
    )


    customer_geo["lon"] = (
        customer_geo["city"]
        .map(
            lambda x:
            coordinates.get(
                str(x).strip(),
                {}
            ).get("lon")
        )
    )


    customer_geo = customer_geo.dropna(
        subset=[
            "lat",
            "lon"
        ]
    ).copy()


    # =====================================================
    # COMBINED MAP
    # =====================================================

    map_points = pd.concat(
        [
            store_geo[
                [
                    "lat",
                    "lon"
                ]
            ],
            customer_geo[
                [
                    "lat",
                    "lon"
                ]
            ]
        ],
        ignore_index=True
    )


    if len(map_points) > 0:

        center_lat = (
            map_points["lat"]
            .mean()
        )

        center_lon = (
            map_points["lon"]
            .mean()
        )


        metric_column_map = {
            "Transaction Count":
                "Transactions",
            "Revenue":
                "Revenue",
            "Profit":
                "Profit"
        }


        size_column = metric_column_map[
            map_metric
        ]


        fig = go.Figure()


        # =================================================
        # CUSTOMER DEMAND
        # =================================================

        if len(customer_geo) > 0:

            max_customer_value = (
                customer_geo[size_column]
                .max()
            )


            if max_customer_value > 0:

                customer_sizeref = (
                    2.0
                    * max_customer_value
                    / (32 ** 2)
                )

            else:

                customer_sizeref = 1


            fig.add_trace(
                go.Scattermap(
                    lat=customer_geo["lat"],
                    lon=customer_geo["lon"],
                    mode="markers",
                    name="Customer Demand",
                    hovertemplate=(
                        "<b>Customer Demand</b>"
                        "<br>City: %{customdata[0]}"
                        "<br>Transactions: %{customdata[1]:,.0f}"
                        "<br>Revenue: $%{customdata[2]:,.2f}"
                        "<br>Profit: $%{customdata[3]:,.2f}"
                        "<extra></extra>"
                    ),
                    customdata=customer_geo[
                        [
                            "city",
                            "Transactions",
                            "Revenue",
                            "Profit"
                        ]
                    ].values,
                    marker=dict(
                        size=customer_geo[
                            size_column
                        ],
                        sizemode="area",
                        sizeref=customer_sizeref,
                        sizemin=9,
                        symbol="circle",
                        color="#f472b6",
                        opacity=0.55,
                        allowoverlap=True
                    )
                )
            )


        # =================================================
        # STORE LOCATIONS
        # =================================================

        if len(store_geo) > 0:

            fig.add_trace(
                go.Scattermap(
                    lat=store_geo["lat"],
                    lon=store_geo["lon"],
                    mode="markers",
                    name="Existing Stores",
                    hovertemplate=(
                        "<b>Existing Store Location</b>"
                        "<br>City: %{customdata[0]}"
                        "<br>Transactions: %{customdata[1]:,.0f}"
                        "<br>Revenue: $%{customdata[2]:,.2f}"
                        "<br>Profit: $%{customdata[3]:,.2f}"
                        "<extra></extra>"
                    ),
                    customdata=store_geo[
                        [
                            "city",
                            "Transactions",
                            "Revenue",
                            "Profit"
                        ]
                    ].values,
                    marker=dict(
                        size=18,
                        symbol="pharmacy",
                        color="#22d3ee",
                        opacity=1
                    )
                )
            )


        # =================================================
        # MAP LAYOUT
        # =================================================

        fig.update_layout(
            height=560,
            margin=dict(
                l=0,
                r=0,
                t=0,
                b=0
            ),
            paper_bgcolor="#111827",
            map=dict(
                style="dark",
                center=dict(
                    lat=center_lat,
                    lon=center_lon
                ),
                zoom=4.5
            ),
            legend=dict(
                orientation="h",
                yanchor="bottom",
                y=0.015,
                xanchor="left",
                x=0.015,
                bgcolor="rgba(17,24,39,0.88)",
                bordercolor="#273449",
                borderwidth=1,
                font=dict(
                    color="#ffffff"
                )
            )
        )


        st.plotly_chart(
            fig,
            use_container_width=True,
            config={
                "displayModeBar": True,
                "scrollZoom": True,
                "displaylogo": False
            }
        )


        st.caption(
            "Cyan pharmacy markers represent existing stores. "
            "Pink circles represent customer demand by city. "
            "Circle size represents the selected demand metric. "
            "Locations are city-level coordinates."
        )


    else:

        st.warning(
            "No valid store or customer city coordinates "
            "could be resolved."
        )


    # =====================================================
    # COVERAGE ANALYSIS
    # =====================================================

    section_header(
        "Market Coverage Analysis"
    )


    coverage_data = nearest_store_analysis(
        customer_geo=(
            (
                customer_geo
                if demand_view == "All Customer Cities"
                else (
                    (
                        filtered_sales
                        .groupby(
                            "customer_city",
                            dropna=False
                        )
                        .agg(
                            Transactions=(
                                "sale_id",
                                "nunique"
                            ),
                            Revenue=(
                                "total_amount",
                                "sum"
                            ),
                            Profit=(
                                "profit",
                                "sum"
                            )
                        )
                        .reset_index()
                        .rename(
                            columns={
                                "customer_city":
                                    "city"
                            }
                        )
                    )
                    .assign(
                        **{
                            "Profit Margin":
                            lambda x:
                            (
                                x["Profit"]
                                / x["Revenue"]
                                * 100
                            ).where(
                                x["Revenue"] != 0
                            ).fillna(0)
                        }
                    )
                )
            )
        ),
        store_geo=store_geo
    )


    if len(coverage_data) > 0:

        st.markdown(
            """
            <div style="
                color:#ffffff;
                font-size:12px;
                margin-bottom:12px;
            ">
                Customer cities are evaluated against their nearest
                existing store using city-level geographic distance.
            </div>
            """,
            unsafe_allow_html=True
        )


        coverage_col1, coverage_col2 = st.columns(2)


        with coverage_col1:

            distance_threshold = st.slider(
                "Minimum distance to consider underserved (km)",
                min_value=5,
                max_value=100,
                value=25,
                step=5,
                key="coverage_distance_threshold"
            )


        with coverage_col2:

            demand_threshold = st.slider(
                "Minimum customer transactions",
                min_value=1,
                max_value=max(
                    1,
                    int(
                        coverage_data[
                            "Customer Transactions"
                        ].max()
                    )
                ),
                value=min(
                    10,
                    max(
                        1,
                        int(
                            coverage_data[
                                "Customer Transactions"
                            ].max()
                        )
                    )
                ),
                step=1,
                key="coverage_demand_threshold"
            )


        coverage_data[
            "Potential Expansion"
        ] = (
            (
                coverage_data["Distance (km)"]
                >= distance_threshold
            )
            &
            (
                coverage_data["Customer Transactions"]
                >= demand_threshold
            )
        )


        expansion_data = (
            coverage_data[
                coverage_data[
                    "Potential Expansion"
                ]
            ]
            .copy()
            .sort_values(
                [
                    "Customer Transactions",
                    "Distance (km)"
                ],
                ascending=[
                    False,
                    False
                ]
            )
        )


        # =================================================
        # EXPANSION KPIs
        # =================================================

        k1, k2, k3 = st.columns(3)


        expansion_count = len(
            expansion_data
        )


        expansion_transactions = (
            expansion_data[
                "Customer Transactions"
            ].sum()
            if expansion_count > 0
            else 0
        )


        expansion_revenue = (
            expansion_data[
                "Customer Revenue"
            ].sum()
            if expansion_count > 0
            else 0
        )


        expansion_cards = [
            (
                "Potential Expansion Areas",
                f"{expansion_count:,.0f}",
                "Customer cities meeting both conditions"
            ),
            (
                "Demand in These Areas",
                f"{expansion_transactions:,.0f}",
                "Customer transactions"
            ),
            (
                "Associated Revenue",
                money(expansion_revenue),
                "Revenue generated from these cities"
            )
        ]


        for col, card in zip(
            [k1, k2, k3],
            expansion_cards
        ):

            with col:

                st.markdown(
                    f"""
                    <div class="kpi-card">
                        <div class="kpi-label">
                            {card[0]}
                        </div>
                        <div class="kpi-value">
                            {card[1]}
                        </div>
                        <div class="kpi-sub">
                            {card[2]}
                        </div>
                    </div>
                    """,
                    unsafe_allow_html=True
                )


        # =================================================
        # EXPANSION TABLE
        # =================================================

        chart_title(
            "Potential Expansion Areas",
            "High customer demand combined with greater distance from the nearest existing store"
        )


        if len(expansion_data) > 0:

            expansion_display = (
                expansion_data[
                    [
                        "Customer City",
                        "Customer Transactions",
                        "Customer Revenue",
                        "Customer Profit",
                        "Nearest Store",
                        "Distance (km)"
                    ]
                ]
                .copy()
            )


            expansion_display[
                "Customer Revenue"
            ] = expansion_display[
                "Customer Revenue"
            ].round(2)


            expansion_display[
                "Customer Profit"
            ] = expansion_display[
                "Customer Profit"
            ].round(2)


            expansion_display[
                "Distance (km)"
            ] = expansion_display[
                "Distance (km)"
            ].round(1)


            st.dataframe(
                expansion_display,
                use_container_width=True,
                hide_index=True
            )


        else:

            st.info(
                "No customer cities currently meet "
                "the selected expansion criteria."
            )


        # =================================================
        # NEAREST STORE ANALYSIS
        # =================================================

        chart_title(
            "Customer City Coverage",
            "Distance and activity of the nearest existing store"
        )


        coverage_display = (
            coverage_data[
                [
                    "Customer City",
                    "Customer Transactions",
                    "Customer Revenue",
                    "Nearest Store",
                    "Nearest Store Transactions",
                    "Distance (km)"
                ]
            ]
            .copy()
            .sort_values(
                "Distance (km)",
                ascending=False
            )
        )


        coverage_display[
            "Customer Revenue"
        ] = coverage_display[
            "Customer Revenue"
        ].round(2)


        coverage_display[
            "Distance (km)"
        ] = coverage_display[
            "Distance (km)"
        ].round(1)


        st.dataframe(
            coverage_display,
            use_container_width=True,
            hide_index=True
        )


    else:

        st.info(
            "Coverage analysis requires both customer "
            "and store city coordinates."
        )


    # =====================================================
    # STORE OVERLAP ANALYSIS
    # =====================================================

    section_header(
        "Store Proximity & Possible Overlap"
    )


    st.markdown(
        """
        <div style="
            color:#ffffff;
            font-size:12px;
            margin-bottom:12px;
        ">
            Identify stores located close to each other and compare
            their transaction activity. This is a city-level analysis
            when exact store coordinates are not available.
        </div>
        """,
        unsafe_allow_html=True
    )


    # =====================================================
    # STORE-LEVEL SALES
    # =====================================================

    store_details = (
        filtered_sales
        .groupby(
            [
                "store_id",
                "store_name",
                "store_city"
            ],
            dropna=False
        )
        .agg(
            Transactions=(
                "sale_id",
                "nunique"
            ),
            Revenue=(
                "total_amount",
                "sum"
            ),
            Profit=(
                "profit",
                "sum"
            )
        )
        .reset_index()
    )


    store_details = store_details.merge(
        store_geo[
            [
                "city",
                "lat",
                "lon"
            ]
        ],
        left_on="store_city",
        right_on="city",
        how="left"
    )


    store_details = store_details.drop(
        columns=[
            "city"
        ],
        errors="ignore"
    )


    overlap_data = store_overlap_analysis(
        store_details
    )


    if len(overlap_data) > 0:

        overlap_col1, overlap_col2 = st.columns(2)


        with overlap_col1:

            overlap_distance = st.slider(
                "Consider stores close within (km)",
                min_value=1,
                max_value=50,
                value=10,
                step=1,
                key="overlap_distance"
            )


        with overlap_col2:

            low_activity_threshold = st.slider(
                "Low transaction threshold per store",
                min_value=1,
                max_value=max(
                    1,
                    int(
                        store_details[
                            "Transactions"
                        ].max()
                    )
                ),
                value=min(
                    20,
                    max(
                        1,
                        int(
                            store_details[
                                "Transactions"
                            ].max()
                        )
                    )
                ),
                step=1,
                key="low_activity_threshold"
            )


        overlap_data["Potential Overlap"] = (
            (
                overlap_data[
                    "Distance (km)"
                ]
                <= overlap_distance
            )
            &
            (
                overlap_data[
                    "Transactions A"
                ]
                <= low_activity_threshold
            )
            &
            (
                overlap_data[
                    "Transactions B"
                ]
                <= low_activity_threshold
            )
        )


        potential_overlap = (
            overlap_data[
                overlap_data[
                    "Potential Overlap"
                ]
            ]
            .copy()
            .sort_values(
                [
                    "Distance (km)",
                    "Combined Transactions"
                ],
                ascending=[
                    True,
                    True
                ]
            )
        )


        overlap_k1, overlap_k2 = st.columns(2)


        with overlap_k1:

            st.markdown(
                f"""
                <div class="kpi-card">
                    <div class="kpi-label">
                        Potential Store Overlaps
                    </div>
                    <div class="kpi-value">
                        {len(potential_overlap):,.0f}
                    </div>
                    <div class="kpi-sub">
                        Pairs meeting distance + activity conditions
                    </div>
                </div>
                """,
                unsafe_allow_html=True
            )


        with overlap_k2:

            total_overlap_transactions = (
                potential_overlap[
                    "Combined Transactions"
                ].sum()
                if len(potential_overlap) > 0
                else 0
            )


            st.markdown(
                f"""
                <div class="kpi-card">
                    <div class="kpi-label">
                        Transactions Across These Pairs
                    </div>
                    <div class="kpi-value">
                        {total_overlap_transactions:,.0f}
                    </div>
                    <div class="kpi-sub">
                        Combined activity of flagged stores
                    </div>
                </div>
                """,
                unsafe_allow_html=True
            )


        chart_title(
            "Potential Store Overlap",
            "Nearby stores with relatively low transaction activity"
        )


        if len(potential_overlap) > 0:

            overlap_display = (
                potential_overlap[
                    [
                        "Store A",
                        "City A",
                        "Transactions A",
                        "Store B",
                        "City B",
                        "Transactions B",
                        "Distance (km)",
                        "Combined Transactions"
                    ]
                ]
                .copy()
            )


            overlap_display[
                "Distance (km)"
            ] = overlap_display[
                "Distance (km)"
            ].round(1)


            st.dataframe(
                overlap_display,
                use_container_width=True,
                hide_index=True
            )


        else:

            st.info(
                "No nearby store pairs currently meet "
                "the selected low-activity criteria."
            )


    else:

        st.info(
            "Store overlap analysis requires at least two "
            "stores with resolvable city coordinates."
        )


# =========================================================
# OTHER PAGES
# =========================================================

elif page == "Customer Loyalty":

    st.title("Customer Loyalty")

    st.write(
        "Customer Loyalty dashboard — coming next."
    )


elif page == "Inventory":

    st.title("Inventory")

    st.write(
        "Inventory dashboard — coming next."
    )


elif page == "Employees":

    st.title("Employees")

    st.write(
        "Employees dashboard — coming next."
    )


elif page == "Suppliers":

    st.title("Suppliers")

    st.write(
        "Suppliers dashboard — coming next."
    )

Overwriting app.py


In [ ]:
import os

os.makedirs("dashboard_data", exist_ok=True)

dataframes = {
    "categories": categories,
    "customers": customers,
    "employees": employees,
    "prescriptions": prescriptions,
    "products": products,
    "stock": stock,
    "suppliers": suppliers,
    "stores": stores,
    "sales": sales,
}

for name, df in dataframes.items():
    df.to_parquet(f"dashboard_data/{name}.parquet", index=False)

print("Dashboard data exported successfully.")

Dashboard data exported successfully.
